#### Imports & Setup

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import sys

# Detect project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

print(f"Project Root: {PROJECT_ROOT}")

# Create directories
dirs = [
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "app",
    PROJECT_ROOT / "notebooks"
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)

print("Project directories created successfully!")

Project Root: C:\Users\tohiba\Desktop\coffee-cupping-qc-dashboard
Project directories created successfully!


#### Define Data Schema (Core Columns)

In [2]:
# Industry-standard columns for coffee cupping (SCA + ECX)

cupping_columns = {
    'sample_id': 'str',                
    'lot_id': 'str',                    
    'supplier_name': 'str',
    'region': 'str',
    'processing_method': 'str',
    'roast_date': 'datetime64[ns]',
    'cupping_date': 'datetime64[ns]',
    
    # SCA Classic Scores (out of 10)
    'aroma': 'float',
    'flavor': 'float',
    'aftertaste': 'float',
    'acidity': 'float',
    'body': 'float',
    'balance': 'float',
    'uniformity': 'float',
    'clean_cup': 'float',
    'sweetness': 'float',
    'overall': 'float',
    
    'total_score': 'float',             
    'quality_tier': 'str',              
    
    # Physical / ECX Parameters
    'defects_per_300g': 'int',
    'moisture_pct': 'float',
    'water_activity': 'float',
    'grade_ecx': 'str',
    
    # Descriptive Notes
    'flavor_notes': 'str',              
    'defect_notes': 'str',
    'general_comments': 'str',
    
    'cupper_name': 'str',
    'last_updated': 'datetime64[ns]'
}

print("Cupping Data Schema Defined")
print(f"Total Columns: {len(cupping_columns)}")

Cupping Data Schema Defined
Total Columns: 28


#### Create Mock Data (Realistic Ethiopian Lots)

In [3]:
np.random.seed(42)

n_samples = 60

data = {
    'sample_id': [f"CUP-{str(i).zfill(4)}" for i in range(1, n_samples+1)],
    'lot_id': [f"ETH-2025-{reg[:3].upper()}-{str(i).zfill(3)}" for i, reg in enumerate(np.random.choice(['Yirgacheffe','Guji','Sidama','Limmu'], n_samples), 1)],
    'supplier_name': np.random.choice(['Daye Bensa', 'Hambela', 'Misty Valley', 'Guji Highland', 'Red Cherry'], n_samples),
    'region': np.random.choice(['Yirgacheffe', 'Guji', 'Sidama', 'Limmu'], n_samples),
    'processing_method': np.random.choice(['Washed', 'Natural', 'Honey'], n_samples, p=[0.5, 0.4, 0.1]),
    'roast_date': pd.date_range(start='2026-04-01', periods=n_samples, freq='D'),
    'cupping_date': datetime.now(),
    
    'aroma': np.round(np.random.uniform(7.5, 9.5, n_samples), 1),
    'flavor': np.round(np.random.uniform(7.8, 9.6, n_samples), 1),
    'aftertaste': np.round(np.random.uniform(7.5, 9.4, n_samples), 1),
    'acidity': np.round(np.random.uniform(7.8, 9.5, n_samples), 1),
    'body': np.round(np.random.uniform(7.5, 9.2, n_samples), 1),
    'balance': np.round(np.random.uniform(7.8, 9.3, n_samples), 1),
    'uniformity': np.random.choice([9.0, 10.0], n_samples, p=[0.15, 0.85]),
    'clean_cup': np.random.choice([9.0, 10.0], n_samples, p=[0.2, 0.8]),
    'sweetness': np.random.choice([9.0, 10.0], n_samples, p=[0.1, 0.9]),
    'overall': np.round(np.random.uniform(8.0, 9.5, n_samples), 1),
}

df = pd.DataFrame(data)

# Calculate Total Score
score_cols = ['aroma','flavor','aftertaste','acidity','body','balance','uniformity','clean_cup','sweetness','overall']
df['total_score'] = df[score_cols].sum(axis=1).round(1)

# Quality Tier
df['quality_tier'] = pd.cut(df['total_score'], 
                           bins=[0, 82, 85, 88, 100], 
                           labels=['Good', 'Very Good', 'Excellent', 'Outstanding'])

df['defects_per_300g'] = np.random.randint(0, 12, n_samples)
df['moisture_pct'] = np.round(np.random.uniform(9.5, 12.1, n_samples), 2)
df['water_activity'] = np.round(np.random.uniform(0.57, 0.68, n_samples), 3)
df['grade_ecx'] = np.where(df['defects_per_300g'] <= 3, 'Grade 1', 'Grade 2')

df['flavor_notes'] = np.random.choice([
    "Jasmine, Bergamot, Lemon Zest", "Blueberry, Chocolate, Bright",
    "Peach, Black Tea, Floral", "Strawberry, Tropical, Winey",
    "Citrus, Honey, Complex"
], n_samples)

df['cupper_name'] = "Aklilu Abera"
df['last_updated'] = datetime.now()

print("Mock Cupping Data Created")
print(f"Total Samples: {len(df)}")
print(f"Average SCA Score: {df['total_score'].mean():.2f}")

Mock Cupping Data Created
Total Samples: 60
Average SCA Score: 89.59


In [5]:
df.head()

,sample_id,lot_id,supplier_name,region,processing_method,roast_date,cupping_date,aroma,flavor,aftertaste,...,overall,total_score,quality_tier,defects_per_300g,moisture_pct,water_activity,grade_ecx,flavor_notes,cupper_name,last_updated
0,CUP-0001,ETH-2025-SID-001,Misty Valley,Yirgacheffe,Natural,2026-04-01,2026-05-23 09:31:41.739453,7.9,8.5,8.5,...,9.2,87.6,Excellent,10,11.87,0.607,Grade 2,"Blueberry, Chocolate, Bright",Aklilu Abera,2026-05-23 09:31:42.296240
1,CUP-0002,ETH-2025-LIM-002,Red Cherry,Guji,Washed,2026-04-02,2026-05-23 09:31:41.739453,7.6,9.4,8.0,...,8.4,86.9,Excellent,9,9.82,0.578,Grade 2,"Citrus, Honey, Complex",Aklilu Abera,2026-05-23 09:31:42.296240
2,CUP-0003,ETH-2025-YIR-003,Daye Bensa,Sidama,Washed,2026-04-03,2026-05-23 09:31:41.739453,8.7,8.9,8.6,...,9.0,90.0,Outstanding,0,10.11,0.653,Grade 1,"Jasmine, Bergamot, Lemon Zest",Aklilu Abera,2026-05-23 09:31:42.296240
3,CUP-0004,ETH-2025-SID-004,Hambela,Guji,Washed,2026-04-04,2026-05-23 09:31:41.739453,8.9,9.2,7.6,...,9.1,87.3,Excellent,5,9.93,0.600,Grade 2,"Jasmine, Bergamot, Lemon Zest",Aklilu Abera,2026-05-23 09:31:42.296240
4,CUP-0005,ETH-2025-SID-005,Guji Highland,Yirgacheffe,Washed,2026-04-05,2026-05-23 09:31:41.739453,7.5,8.7,7.6,...,8.9,90.4,Outstanding,5,9.98,0.669,Grade 2,"Peach, Black Tea, Floral",Aklilu Abera,2026-05-23 09:31:42.296240


#### Save the Data

In [4]:
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

data_folder = PROJECT_ROOT / "data"
processed_folder = data_folder / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

df.to_csv(data_folder / "mock_cupping_data.csv", index=False)
df.to_pickle(processed_folder / "cupping_data.pkl")
df.to_excel(processed_folder / "cupping_data.xlsx", index=False)

print("Cupping data saved successfully!")

Cupping data saved successfully!


#### Summary

In [6]:
print("=== CUPPING QC PROJECT ===\n")
print(f"Total Cupping Records : {len(df)}")
print(f"Score Range           : {df['total_score'].min()} - {df['total_score'].max()}")
print(f"Outstanding Lots      : {(df['total_score'] >= 88).sum()}")
print(f"Grade 1 Samples       : {(df['grade_ecx'] == 'Grade 1').sum()}")

=== CUPPING QC PROJECT ===

Total Cupping Records : 60
Score Range           : 85.7 - 93.1
Outstanding Lots      : 51
Grade 1 Samples       : 16
